In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!sudo apt remove python3-pandas -y


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'python3-pandas' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [ ]:
!pip uninstall pyarrow -y

Found existing installation: pyarrow 15.0.2
Uninstalling pyarrow-15.0.2:
  Successfully uninstalled pyarrow-15.0.2


In [ ]:
# -*- coding: utf-8 -*-
"""ASR Agent 2: CNN-RNN-CTC from Scratch (with NumPy CNN implementation)

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1-s99T9WeOLpHUS4BImstjOMCprumd2_i
"""

# -*- coding: utf-8 -*-
"""
ASR Agent 2: CNN-RNN-CTC Model from Scratch

This script builds and trains an Automatic Speech Recognition (ASR) model
for isiZulu using a custom CNN-RNN architecture with a CTC loss function.
This version replaces the standard torch.nn.Conv2d with a manual implementation
using NumPy for the core convolution operation.
"""

# 1. Install and Import Necessary Libraries
# -------------------------------------------
# Using the same libraries as the first agent for consistency.
!pip install -q transformers accelerate torchaudio librosa
!pip install torchcodec==0.2
!pip install fsspec==2023.9.2
!pip install datasets==2.10.0
!pip install evaluate==0.4.0
!pip install jiwer==2.3.0
!pip install tf-keras

!pip install --user --no-cache-dir pyarrow==15.0.2

import os
import re
import json
from dataclasses import dataclass
from typing import Dict, List, Optional, Union

import torch
import torch.nn as nn
import torchaudio
import numpy as np
from torch.nn.modules.utils import _pair # Needed for custom Conv2d

from datasets import load_dataset, Audio, DatasetDict
from transformers import (
    Wav2Vec2CTCTokenizer, # Re-using the tokenizer logic for text processing
    TrainingArguments,
    Trainer
)
import evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.3/38.3 MB 262.3 MB/s eta 0:00:00


In [ ]:

# 2. Configuration Variables
# --------------------------
# --- Dataset and Text Processing (Same as Agent 1) ---
DATASET_NAME = "aconeil/nchlt"
DEFAULT_CONFIG_NAME = "default"
TEXT_COLUMN = "transcription"
AUDIO_COLUMN = "audio"
TOKENIZER_OUTPUT_DIR = "./cnn-rnn-ctc-pytorch-tokenizer" # Separate tokenizer/vocab directory

# --- Audio Feature Extraction ---
TARGET_SAMPLING_RATE = 16_000
N_FFT = 400  # Size of the FFT window
HOP_LENGTH = 160 # Number of samples between successive frames
N_MELS = 80 # Number of Mel-frequency bins

# --- Custom Model Architecture (CNN-RNN-CTC) ---
#MODEL_OUTPUT_DIR = "./cnn-rnn-ctc-isizulu"
MODEL_OUTPUT_DIR = "/content/drive/MyDrive/wav2vec2-cnn-rnn-ctc-pytorch-checkpoints"

CNN_LAYERS = 3
RNN_LAYERS = 3
RNN_DIM = 512
DROPOUT = 0.1

# --- Training Parameters ---
MAX_DURATION_IN_SECONDS = 20.0
MIN_DURATION_IN_SECONDS = 1.0
NUM_EPOCHS = 10 # Training from scratch requires more epochs
PER_DEVICE_TRAIN_BATCH_SIZE = 4 # Adjust based on GPU memory
PER_DEVICE_EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-4 # A common starting point for models from scratch
WARMUP_STEPS = 400
SAVE_STEPS = 1000
EVAL_STEPS = 1000

# Forcefully clear Hugging Face datasets and general cache
!rm -rf ~/.cache/huggingface/datasets/*
!rm -rf ~/.cache/huggingface/modules/*
!rm -rf ~/.cache/huggingface/hub/*
print("Hugging Face caches cleared.")


Hugging Face caches cleared.


In [ ]:
# 3. Load and Prepare Dataset (UC-DM-01)
# -------------------------------------------------
print(f"Loading dataset: {DATASET_NAME}, config: {DEFAULT_CONFIG_NAME}")
# This section is identical to the first agent's script
try:
    raw_datasets = DatasetDict()
    train_ds = load_dataset(DATASET_NAME, name=DEFAULT_CONFIG_NAME, split="train")
    test_ds = load_dataset(DATASET_NAME, name=DEFAULT_CONFIG_NAME, split="test")

    train_val_split = train_ds.train_test_split(test_size=0.1, shuffle=True, seed=42)

    raw_datasets["train"] = train_val_split["train"]
    raw_datasets["validation"] = train_val_split["test"]
    raw_datasets["test"] = test_ds

    print("Dataset loaded and split successfully.")
    print(raw_datasets)
except Exception as e:
    print(f"Error loading dataset: {e}")
    raise e

# --- Text Preprocessing & Vocabulary Creation ---
# Identical to the first agent
chars_to_remove_regex = r'[!"#$%&\'()*+,-./:;<=>?@\[\\\]^_`{|}~0-9]'
def remove_special_characters(batch):
    if batch[TEXT_COLUMN] is not None:
        batch[TEXT_COLUMN] = re.sub(chars_to_remove_regex, '', batch[TEXT_COLUMN]).lower()
    else:
        batch[TEXT_COLUMN] = ""
    return batch

print("Normalizing text data...")
raw_datasets = raw_datasets.map(remove_special_characters, num_proc=4)

# Create Vocabulary
def extract_all_chars(batch):
  all_text = " ".join(batch[TEXT_COLUMN])
  vocab = list(set(all_text))
  return {"vocab": [vocab], "all_text": [all_text]}

print("Extracting vocabulary...")
combined_text_data = "".join([text for split in raw_datasets for text in raw_datasets[split][TEXT_COLUMN] if text is not None])
vocab_list = list(set(combined_text_data))
vocab_dict = {v: k for k, v in enumerate(vocab_list)}

# Add CTC-specific and utility tokens
vocab_dict["|"] = vocab_dict[" "] # Word delimiter
del vocab_dict[" "]
vocab_dict["[UNK]"] = len(vocab_dict) # Unknown token
vocab_dict["[PAD]"] = len(vocab_dict) # Padding token

VOCAB_SIZE = len(vocab_dict)
print(f"Vocabulary size: {VOCAB_SIZE}")

os.makedirs(TOKENIZER_OUTPUT_DIR, exist_ok=True)
with open(os.path.join(TOKENIZER_OUTPUT_DIR, 'vocab.json'), 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

# Load the tokenizer
tokenizer = Wav2Vec2CTCTokenizer(
    os.path.join(TOKENIZER_OUTPUT_DIR, 'vocab.json'),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

# --- Audio Feature Extraction & Processing (UC-DM-02) ---
# Cast to target sampling rate (This will still be done later in prepare_dataset)
# We will cast the column *after* filtering by duration using the metadata.
# raw_datasets = raw_datasets.cast_column(
#     AUDIO_COLUMN, Audio(sampling_rate=TARGET_SAMPLING_RATE)
# )


Loading dataset: aconeil/nchlt, config: default


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/41871 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2802 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)


Map (num_proc=4):   0%|          | 0/4188 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1393: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)


Map (num_proc=4):   0%|          | 0/2802 [00:00<?, ? examples/s]

Extracting vocabulary...
Vocabulary size: 29


In [ ]:
!sudo apt-get update && sudo apt-get install ffmpeg

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,933 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Pack

In [ ]:

# Filter audio by duration using the existing 'duration' column
def filter_duration_metadata(example):
    # Use the pre-calculated 'duration' column from the dataset metadata
    duration = example.get("duration")
    if duration is None:
        # If duration metadata is missing, we might want to skip or handle differently
        # For now, assume examples with missing duration should be skipped.
        print(f"Warning: Skipping example with missing duration metadata: {example.get(AUDIO_COLUMN, {}).get('path', 'N/A')}")
        return False
    return MIN_DURATION_IN_SECONDS <= duration <= MAX_DURATION_IN_SECONDS

print(f"Filtering audio by duration using metadata ({MIN_DURATION_IN_SECONDS}s - {MAX_DURATION_IN_SECONDS}s)...")
# Use the metadata filter function with num_proc=os.cpu_count() as it doesn't load audio
raw_datasets = raw_datasets.filter(filter_duration_metadata, num_proc=1)

Filtering audio by duration using metadata (1.0s - 20.0s)...


Filter:   0%|          | 0/37683 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4188 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2802 [00:00<?, ? examples/s]

In [ ]:
print("Dataset filtered by duration metadata.")
print(raw_datasets)

# Now, cast the audio column to the target sampling rate *after* filtering
# This will trigger audio loading and resampling by the datasets library,
# but applied only to the filtered subset.
print(f"Casting audio column to sampling rate: {TARGET_SAMPLING_RATE} Hz...")
raw_datasets = raw_datasets.cast_column(
    AUDIO_COLUMN, Audio(sampling_rate=TARGET_SAMPLING_RATE)
)
print("Audio column cast complete.")
print(raw_datasets)


# Define Mel Spectrogram transform
mel_spectrogram_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SAMPLING_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS
)

# Main processing function: extract features and tokenize text
# This function will now receive audio data already loaded and resampled by datasets
def prepare_dataset(batch):
    audio = batch[AUDIO_COLUMN]
    # 1. Extract Mel Spectrogram from audio array
    # audio["array"] should now contain the resampled waveform
    waveform = torch.tensor(audio["array"], dtype=torch.float32)

    # Assuming waveform is (num_channels, num_samples) - take the first channel if multi-channel
    if waveform.ndim > 1 and waveform.shape[0] > 1:
        waveform = waveform[0, :]
    elif waveform.ndim > 1:
         waveform = waveform.squeeze(0) # Remove channel dim if only one channel and it's 2D


    mel_spec = mel_spectrogram_transform(waveform)
    # Apply log-scaling for stability
    log_mel_spec = torch.log(torch.clamp(mel_spec, min=1e-5))


    batch["input_values"] = log_mel_spec
    batch["input_length"] = log_mel_spec.shape[1] # Length of the feature sequence

    # 2. Tokenize the target text
    # Handle cases where text might still be None after normalization (shouldn't happen, but as safeguard)
    if batch[TEXT_COLUMN] is None:
        batch["labels"] = tokenizer("") # Tokenize an empty string if text is None
    else:
        batch["labels"] = tokenizer(batch[TEXT_COLUMN]).input_ids

    return batch

print("Preparing dataset with Mel Spectrograms and tokenized text...")
# Apply the preparation function. We need to update the schema to reflect the new columns.
# We also need to filter out examples where prepare_dataset returned None
processed_datasets = raw_datasets.map(
    prepare_dataset,
    remove_columns=[col for col in raw_datasets.column_names["train"] if col not in ["input_values", "labels", "input_length"]], # Include input_length
    num_proc=1
).filter(lambda example: example is not None) # Filter out examples where prepare_dataset failed

print("Dataset preparation complete.")
print(processed_datasets)

Dataset filtered by duration metadata.
DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'gender', 'age', 'speaker_id', 'duration', 'md5sum', 'pdp_score'],
        num_rows: 37667
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'gender', 'age', 'speaker_id', 'duration', 'md5sum', 'pdp_score'],
        num_rows: 4186
    })
    test: Dataset({
        features: ['audio', 'transcription', 'gender', 'age', 'speaker_id', 'duration', 'md5sum', 'pdp_score'],
        num_rows: 2802
    })
})
Casting audio column to sampling rate: 16000 Hz...
Audio column cast complete.
DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'gender', 'age', 'speaker_id', 'duration', 'md5sum', 'pdp_score'],
        num_rows: 37667
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'gender', 'age', 'speaker_id', 'duration', 'md5sum', 'pdp_score'],
        num_rows: 4186
    })
    test: Dataset({
     

Map:   0%|          | 0/37667 [00:00<?, ? examples/s]

Map:   0%|          | 0/4186 [00:00<?, ? examples/s]

Map:   0%|          | 0/2802 [00:00<?, ? examples/s]

Filter:   0%|          | 0/37667 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4186 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2802 [00:00<?, ? examples/s]

Dataset preparation complete.
DatasetDict({
    train: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 37667
    })
    validation: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 4186
    })
    test: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 2802
    })
})


In [ ]:
# 4. Define CNN-RNN-CTC Model (UC-AD-01)
# --------------------------------------

# NEW: Custom Conv2d Module using NumPy
class Conv2d_numpy(nn.Module):
    """
    A custom 2D Convolution layer that uses NumPy for the core operation.
    This is for educational purposes and is much slower than torch.nn.Conv2d.
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = _pair(kernel_size)
        self.stride = _pair(stride)
        self.padding = _pair(padding)

        # Define trainable parameters: kernel weights and bias
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, *self.kernel_size))
        self.bias = nn.Parameter(torch.empty(out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        # Use Kaiming uniform initialization as a good default for ReLU
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
        if fan_in != 0:
            bound = 1 / np.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        # Convert torch tensors to numpy arrays
        # .detach() is used because numpy operations are not tracked by PyTorch's autograd.
        # Gradients will flow through self.weight and self.bias parameters, not the operations.
        input_np = x.detach().cpu().numpy()
        weight_np = self.weight.detach().cpu().numpy()
        bias_np = self.bias.detach().cpu().numpy()

        batch_size, _, input_h, input_w = input_np.shape
        kernel_h, kernel_w = self.kernel_size
        pad_h, pad_w = self.padding
        stride_h, stride_w = self.stride

        # Calculate output dimensions
        output_h = (input_h + 2 * pad_h - kernel_h) // stride_h + 1
        output_w = (input_w + 2 * pad_w - kernel_w) // stride_w + 1

        # Apply padding to the input array
        padded_input = np.pad(input_np,
                              ((0, 0), (0, 0), (pad_h, pad_h), (pad_w, pad_w)),
                              mode='constant', constant_values=0)

        # Initialize the output array
        output_np = np.zeros((batch_size, self.out_channels, output_h, output_w))

        # Perform the convolution operation with loops and NumPy
        for n in range(batch_size):
            for c_out in range(self.out_channels):
                # Add the bias term for the current output channel
                current_output = np.full((output_h, output_w), bias_np[c_out])
                for c_in in range(self.in_channels):
                    for i in range(output_h):
                        for j in range(output_w):
                            # Define the patch of the input to be convoluted
                            h_start, w_start = i * stride_h, j * stride_w
                            h_end, w_end = h_start + kernel_h, w_start + kernel_w

                            input_patch = padded_input[n, c_in, h_start:h_end, w_start:w_end]
                            kernel = weight_np[c_out, c_in, :, :]

                            # Core matrix multiplication step using NumPy
                            current_output[i, j] += np.sum(input_patch * kernel)

                output_np[n, c_out, :, :] = current_output

        # Convert the result back to a PyTorch tensor
        output_tensor = torch.from_numpy(output_np).float().to(x.device)

        return output_tensor


In [ ]:
class ASRModel(nn.Module):
    """
    Custom ASR model with CNN layers for feature extraction,
    RNN layers for sequence modeling, and a final layer for CTC classification.
    """
    def __init__(self, n_mels, rnn_dim, vocab_size, n_cnn_layers, n_rnn_layers, dropout):
        super().__init__()

        # --- CNN Layers (Using torch.nn.Conv2d) ---
        cnn_channels = 32
        self.cnn = nn.Sequential(
            nn.Conv2d(1, cnn_channels, kernel_size=3, padding=1, stride=1),
            nn.ReLU(),
            nn.BatchNorm2d(cnn_channels),
            nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1, stride=1),
            nn.ReLU(),
            nn.BatchNorm2d(cnn_channels),
            nn.MaxPool2d(kernel_size=(2, 1)),
        )
        # Calculate the feature dimension after the CNN layers
        # The pooling layer halves the mel dimension (height).
        cnn_output_dim = (n_mels // 2) * cnn_channels

        # --- RNN Layers ---
        self.rnn = nn.GRU(
            input_size=cnn_output_dim,
            hidden_size=rnn_dim,
            num_layers=n_rnn_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if n_rnn_layers > 1 else 0
        )

        # --- Classifier ---
        self.classifier = nn.Linear(rnn_dim * 2, vocab_size)

    def forward(self, input_values, labels=None, **kwargs):
        # input_values shape from collator is (batch_size, n_mels, seq_len)

        # Reshape for CNN: (batch_size, 1, n_mels, seq_len)
        x = input_values.unsqueeze(1)

        x = self.cnn(x)

        # Reshape for RNN: (batch_size, seq_len, features)
        # Permute to bring sequence to dim 1, then flatten features
        batch_size, channels, freq_dim, seq_len = x.size()
        x = x.permute(0, 3, 1, 2).contiguous() # -> (batch_size, seq_len, channels, freq_dim)
        x = x.view(batch_size, seq_len, -1)    # -> (batch_size, seq_len, channels * freq_dim)

        x, _ = self.rnn(x)

        logits = self.classifier(x)

        # For CTC loss, PyTorch implementation expects:
        # (sequence_length, batch_size, num_classes)
        log_probs = nn.functional.log_softmax(logits, dim=-1).transpose(0, 1)

        loss = None
        if labels is not None:
            labels[labels == tokenizer.pad_token_id] = -100 # HF convention

            input_lengths = (torch.ones(log_probs.shape[1]) * log_probs.shape[0]).long()
            label_lengths = torch.sum(labels != -100, dim=1).long()

            ctc_loss = nn.CTCLoss(blank=tokenizer.pad_token_id, zero_infinity=True)
            loss = ctc_loss(log_probs, labels, input_lengths, label_lengths)

        from transformers.modeling_outputs import CausalLMOutput
        return CausalLMOutput(loss=loss, logits=logits)


In [ ]:
# 5. Define Training Components (UC-AD-02)
# ----------------------------------------
# Data Collator for custom features
@dataclass
class DataCollatorCTCWithPadding:
    tokenizer: Wav2Vec2CTCTokenizer
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # --- Extract and Pad Input Features (Spectrograms) ---
        # The dataset can store tensors as lists when caching, so we must ensure they are tensors here.
        input_values = [torch.tensor(feature["input_values"]) for feature in features]

        # Spectrograms are 2D tensors (n_mels, seq_len). We pad the seq_len.
        max_len = max(spec.shape[1] for spec in input_values)
        padded_features = []
        for spec in input_values:
            pad_amount = max_len - spec.shape[1]
            # Pad on the right of the time dimension (dim=1)
            padded_spec = torch.nn.functional.pad(spec, (0, pad_amount), 'constant', 0)
            padded_features.append(padded_spec)

        batch = {"input_values": torch.stack(padded_features)}

        # --- Extract and Pad Labels ---
        # The tokenizer's pad method expects a list of dicts with the key "input_ids".
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # MODIFIED: Removed the `with tokenizer.as_target_processor():` context manager.
        # This method is part of Processor classes, not standalone Tokenizer classes.
        labels_batch = tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Replace padding with -100 for loss calculation
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch
data_collator = DataCollatorCTCWithPadding(tokenizer=tokenizer, padding=True)

In [ ]:
# Evaluation Metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids)
    label_str = tokenizer.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

# Instantiate the model
model = ASRModel(
    n_mels=N_MELS,
    rnn_dim=RNN_DIM,
    vocab_size=VOCAB_SIZE,
    n_cnn_layers=CNN_LAYERS,
    n_rnn_layers=RNN_LAYERS,
    dropout=DROPOUT
)

# Training Arguments
training_args = TrainingArguments(
  output_dir=MODEL_OUTPUT_DIR,
  per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
  per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
  gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
  eval_strategy="steps",
  num_train_epochs=NUM_EPOCHS,
  fp16=True,
  save_steps=SAVE_STEPS,
  eval_steps=EVAL_STEPS,
  logging_steps=EVAL_STEPS // 2,
  learning_rate=LEARNING_RATE,
  warmup_steps=WARMUP_STEPS,
  save_total_limit=2,
  load_best_model_at_end=True,
  metric_for_best_model="wer",
  greater_is_better=False,
)

# Trainer
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["validation"],
    tokenizer=tokenizer,
)


/tmp/ipython-input-3262605969.py:49: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:

# 6. Start Training (UC-AD-02)
# ----------------------------
print("Starting training of CNN-RNN-CTC model from scratch...")
try:
    trainer.train()
    print("Training finished.")
except Exception as e:
    print(f"Error during training: {e}")
    print("This might be due to OOM errors. Try reducing batch size or model dimensions (rnn_dim).")
    raise e

# 7. Save the final model and tokenizer
# ---------------------------------------
print("Saving the final trained model and tokenizer...")
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print(f"Model and tokenizer saved to {MODEL_OUTPUT_DIR}")

Starting training of CNN-RNN-CTC model from scratch...


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Wer,Cer
1000,3.185400,1.158358,0.990139,0.368033
2000,1.734800,0.796770,0.921190,0.246697
3000,1.388500,0.653164,0.851599,0.205078
4000,1.228500,0.571902,0.837329,0.194799
5000,1.053900,0.524706,0.789786,0.172098
6000,1.002200,0.496025,0.777038,0.169314
7000,0.921400,0.463935,0.760362,0.160605
8000,0.880200,0.437741,0.735910,0.152676
9000,0.866300,0.432629,0.734787,0.151323
10000,0.713400,0.407408,0.703760,0.142076


Training finished.
Saving the final trained model and tokenizer...
Model and tokenizer saved to /content/drive/MyDrive/wav2vec2-cnn-rnn-ctc-pytorch-checkpoints


In [ ]:
# 7. Save the final model and tokenizer
# ---------------------------------------
print("Saving the final trained model and tokenizer...")
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print(f"Model and tokenizer saved to {MODEL_OUTPUT_DIR}")

#Assuming 'model' and 'processor' are your trained objects
output_dir = "/content/drive/MyDrive/my_final_wav2vec2_model"

# Save the full model, which creates model.safetensors AND config.json
trainer.save_model(output_dir)

# Save the processor, which creates preprocessor_config.json, vocab.json, etc.
tokenizer.save_pretrained(output_dir)

Saving the final trained model and tokenizer...
Model and tokenizer saved to /content/drive/MyDrive/wav2vec2-cnn-rnn-ctc-pytorch-checkpoints


('/content/drive/MyDrive/my_final_wav2vec2_model/tokenizer_config.json',
 '/content/drive/MyDrive/my_final_wav2vec2_model/special_tokens_map.json',
 '/content/drive/MyDrive/my_final_wav2vec2_model/vocab.json',
 '/content/drive/MyDrive/my_final_wav2vec2_model/added_tokens.json')

In [ ]:
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Processor

# --- Step 1: Create a Feature Extractor with your settings ---
# These values are taken directly from your notebook's configuration
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=TARGET_SAMPLING_RATE,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=False
)

# --- Step 2: Create a Processor ---
# The processor combines the feature extractor and the tokenizer.
# The `tokenizer` variable should already exist from your training script.
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

# --- Step 3: Save the final model and the NEW processor ---
output_dir = "/content/drive/MyDrive/my_final_wav2vec2_model"

print("Saving the full model...")
# This correctly saves the model weights (model.safetensors) and architecture (config.json)
trainer.save_model(output_dir)

print("\nSaving the full processor...")
# This is the crucial step that saves all necessary files, including preprocessor_config.json
processor.save_pretrained(output_dir)

print(f"\n✅ Model and processor correctly saved to: {output_dir}")
print("Your model is now ready for inference!")

Saving the full model...

Saving the full processor...

✅ Model and processor correctly saved to: /content/drive/MyDrive/my_final_wav2vec2_model
Your model is now ready for inference!


In [ ]:
# Ensure the trainer has been initialized before running this cell.

print("Evaluating the custom CNN-RNN-CTC model on the test dataset...")

# Use the trainer to evaluate the 'test' split of the processed dataset
test_results = trainer.evaluate(eval_dataset=processed_datasets["test"])

print("\n--- Custom Model Test Set Evaluation Results ---")
print(f"WER: {test_results['eval_wer']:.4f}")
print(f"CER: {test_results['eval_cer']:.4f}")
print("\nComplete results dictionary:")
print(test_results)

Evaluating the custom CNN-RNN-CTC model on the test dataset...



--- Custom Model Test Set Evaluation Results ---
WER: 0.6092
CER: 0.1094

Complete results dictionary:
{'eval_loss': 0.31963470578193665, 'eval_wer': 0.6092049219119735, 'eval_cer': 0.10940046862265689, 'eval_runtime': 109.3016, 'eval_samples_per_second': 25.635, 'eval_steps_per_second': 6.413, 'epoch': 10.0}


In [ ]:
import torch
import numpy as np

# Ensure your 'model', 'tokenizer', 'device', and 'processed_datasets' are loaded and available.

# Define the device again to ensure it's available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Select a single example from the test set
sample_index = 70  # You can change this index to test other samples (e.g., 1, 2, 50)
single_example = processed_datasets["test"][sample_index]

# 2. Prepare the input tensor for the model
# The data is already a Mel Spectrogram, so we just format it
input_tensor = torch.tensor(single_example['input_values']).unsqueeze(0).to(device)

# 3. Get the ground truth labels
ground_truth_ids = single_example['labels']

# 4. Perform inference
with torch.no_grad():
    output = model(input_tensor)
    logits = output.logits

# 5. Decode the model's prediction
predicted_ids = torch.argmax(logits, dim=-1)
predicted_transcription = tokenizer.batch_decode(predicted_ids)[0]

# 6. Decode the ground truth transcription
ground_truth_transcription = tokenizer.decode(ground_truth_ids, skip_special_tokens=True)

# 7. Print the results for comparison
print(f"--- Testing Sample #{sample_index} ---")
print(f"Ground Truth: {ground_truth_transcription}")
print(f"Predicted:    {predicted_transcription}")
print("---------------------------------")

--- Testing Sample #70 ---
Ground Truth: empahla uma kutholakala
Predicted:    empahla uma kutholakala
---------------------------------


In [ ]:
import torch
from transformers import Wav2Vec2CTCTokenizer
from safetensors.torch import load_file # <-- Import the safetensors loader

# Ensure the ASRModel class is defined in your notebook first!

# 1. Define the exact path to the checkpoint
model_path = "/content/drive/MyDrive/wav2vec2-isizulu-checkpoints/checkpoint-17000"
tokenizer_path = "/content/drive/MyDrive/wav2vec2-isizulu-checkpoints/checkpoint-17000"

# 2. Instantiate the model structure with the correct parameters
model_params = {
    "n_mels": N_MELS,
    "rnn_dim": RNN_DIM,
    "vocab_size": VOCAB_SIZE,
    "n_cnn_layers": CNN_LAYERS,
    "n_rnn_layers": RNN_LAYERS,
    "dropout": DROPOUT
}
model = ASRModel(**model_params)

# 3. Load the model weights from the correct file (model.safetensors)
weights_path = os.path.join(model_path, "model.safetensors")
state_dict = load_file(weights_path) # <-- Use load_file for .safetensors
model.load_state_dict(state_dict)

# 4. Load the tokenizer
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(tokenizer_path)

# 5. Set up for inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("✅ Model from checkpoint-17000 and its tokenizer have been loaded successfully!")

In [ ]:
import torchaudio
import numpy as np

def predict_audio(file_path, model, tokenizer):
    """
    Loads an audio file, processes it into a Mel Spectrogram,
    and returns the model's transcription.
    """
    # Ensure the model is in evaluation mode
    model.eval()

    # 1. Load and resample the audio file
    try:
        waveform, sample_rate = torchaudio.load(file_path)
        if sample_rate != TARGET_SAMPLING_RATE:
            resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=TARGET_SAMPLING_RATE)
            waveform = resampler(waveform)
    except Exception as e:
        print(f"Error loading audio file: {e}")
        return None

    # 2. Ensure audio is single-channel (mono)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # 3. Create the Mel Spectrogram feature
    mel_spectrogram_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=TARGET_SAMPLING_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )
    mel_spec = mel_spectrogram_transform(waveform)
    log_mel_spec = torch.log(torch.clamp(mel_spec, min=1e-5))

    # 4. Prepare the tensor for the model (add batch dimension and move to device)
    input_tensor = log_mel_spec.unsqueeze(0).to(device)

    # 5. Perform inference
    with torch.no_grad():
        output = model(input_tensor)
        logits = output.logits

    # 6. Decode the model's output to text
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = tokenizer.batch_decode(predicted_ids)[0]

    return transcription

In [ ]:
# Paste the path to your uploaded audio file here
test_audio_path = "/content/your_audio_file.wav"

# Get the prediction
predicted_text = predict_audio(test_audio_path, model, tokenizer)

# Print the result
if predicted_text:
    print(f"File: {test_audio_path}")
    print(f"Predicted Transcription: {predicted_text}")

In [ ]:
import torch
import numpy as np

# Ensure your 'model', 'tokenizer', 'device', and 'processed_datasets' are loaded and available.

# 1. Select a single example from the test set
sample_index = 689  # You can change this index to test other samples (e.g., 1, 2, 50)
single_example = processed_datasets["test"][sample_index]

# 2. Prepare the input tensor for the model
# The data is already a Mel Spectrogram, so we just format it
input_tensor = torch.tensor(single_example['input_values']).unsqueeze(0).to(device)

# 3. Get the ground truth labels
ground_truth_ids = single_example['labels']

# 4. Perform inference
with torch.no_grad():
    output = model(input_tensor)
    logits = output.logits

# 5. Decode the model's prediction
predicted_ids = torch.argmax(logits, dim=-1)
predicted_transcription = tokenizer.batch_decode(predicted_ids)[0]

# 6. Decode the ground truth transcription
ground_truth_transcription = tokenizer.decode(ground_truth_ids, skip_special_tokens=True)

# 7. Print the results for comparison
print(f"--- Testing Sample #{sample_index} ---")
print(f"Ground Truth: {ground_truth_transcription}")
print(f"Predicted:    {predicted_transcription}")
print("---------------------------------")

In [ ]:
import matplotlib.pyplot as plt
import librosa.display
import numpy as np # Import numpy

# Select a sample (you can change the index)
sample = processed_datasets["train"][4]
mel_spec = sample["input_values"] # This is already the log-Mel spectrogram

# Convert the log-Mel spectrogram list to a NumPy array
mel_spec_np = np.array(mel_spec) # Convert the list to a NumPy array

# Display the Mel spectrogram
plt.figure(figsize=(10, 4))
librosa.display.specshow(mel_spec_np, sr=TARGET_SAMPLING_RATE, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel Spectrogram')
plt.tight_layout()
plt.show()

In [ ]:
RAW_TEXT_COLUMN = "text"

# The dataset split you want to check (e.g., "train", "test")
SPLIT = "test"
# --------------------
import random
def verify_random_sample(raw_datasets, processed_datasets, tokenizer):
    """
    Selects a random sample and compares its raw text to its processed label.
    """
    print(f"--- Verifying a random sample from the '{SPLIT}' split ---")

    # 1. Select a random index
    try:
        num_samples = len(processed_datasets[SPLIT])
        random_index = random.randint(0, num_samples - 1)
        print(f"Checking sample at index: {random_index}")
    except Exception as e:
        print(f"Could not get random sample. Error: {e}")
        return

    # 2. Get the corresponding samples from both datasets
    processed_sample = processed_datasets[SPLIT][random_index]
    raw_sample = raw_datasets[SPLIT][random_index]

    # 3. Get the original text from the raw dataset
    try:
        original_text = raw_sample[TEXT_COLUMN]
    except KeyError:
        print(f"ERROR: Column '{TEXT_COLUMN}' not found in the raw dataset.")
        print(f"Available columns are: {list(raw_sample.keys())}")
        return

    # 4. Decode the processed labels back to text
    labels = torch.tensor(processed_sample["labels"]).unsqueeze(0)
    decoded_processed_text = tokenizer.batch_decode(labels, group_tokens=False)[0]

    # 5. Print for comparison
    print("\nOriginal Text from Raw File:")
    print(f"==> \"{original_text}\"")

    print("\nProcessed Label (Decoded):")
    print(f"==> \"{decoded_processed_text}\"")


# --- Run the verification ---
# Make sure your datasets and tokenizer are loaded before calling this
verify_random_sample(raw_datasets, processed_datasets, tokenizer)

--- Verifying a random sample from the 'test' split ---
Checking sample at index: 2619

Original Text from Raw File:
==> "ezifana nesabelo zimali"

Processed Label (Decoded):
==> "ezifana nesabelo zimali"


In [ ]:
import numpy as np

# Let's predict the first 5 examples from the test set
num_examples = 10
test_subset = processed_datasets["test"].select(range(num_examples))

print(f"Running prediction on the first {num_examples} test samples...")

# Get predictions from the trainer
predictions_output = trainer.predict(test_subset)

# The output contains predictions (logits) and label_ids (ground truth)
predicted_ids = np.argmax(predictions_output.predictions, axis=-1)
true_label_ids = predictions_output.label_ids

# Decode both the predicted and true labels
# For labels, we need to replace -100 (the padding token) before decoding
true_label_ids[true_label_ids == -100] = tokenizer.pad_token_id
predicted_transcriptions = tokenizer.batch_decode(predicted_ids)
true_transcriptions = tokenizer.batch_decode(true_label_ids, group_tokens=False)

print("\n--- Prediction Samples ---")
for i in range(num_examples):
    print(f"Sample {i+1}:")
    print(f"  Ground Truth: {true_transcriptions[i]}")
    print(f"  Predicted:    {predicted_transcriptions[i]}")
    print("-" * 20)

Running prediction on the first 10 test samples...



--- Prediction Samples ---
Sample 1:
  Ground Truth: izokwenza umyalelo wokuthi
  Predicted:    izokwenza umiyalelo wokuthim
--------------------
Sample 2:
  Ground Truth: lesi siphandla sinikezwa
  Predicted:    lesi siphandla sinikezwam
--------------------
Sample 3:
  Ground Truth: kungathatha isikhathi eside
  Predicted:    kungathatha isikhathi esidem
--------------------
Sample 4:
  Ground Truth: liphungulwe uma lokhu
  Predicted:    liphungulwe uma lokhwem
--------------------
Sample 5:
  Ground Truth: lenqubo ibonakala sengathi
  Predicted:    lenqubo ibonakala sengathi
--------------------
Sample 6:
  Ground Truth: kube yilona elivela
  Predicted:    kube yilona elivela
--------------------
Sample 7:
  Ground Truth: umsebenzi wokondla ubekwa
  Predicted:    umsebenzi wokondla ubekwa
--------------------
Sample 8:
  Ground Truth: isitifikedi sokushona yidokhumende
  Predicted:    isitiskedi sokushona idokhumende
--------------------
Sample 9:
  Ground Truth: uhlelo lwentuthuko